# NFL Analytics: Metrics

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from pathlib import Path

pio.renderers.default = "notebook_connected"

OUTPUTS = Path("../../data/outputs")

epa   = pd.read_csv(OUTPUTS / "epa_by_team.csv")
sr    = pd.read_csv(OUTPUTS / "success_rate_by_team.csv")
wpa   = pd.read_csv(OUTPUTS / "wpa_by_team.csv").query("posteam != 'NA'")
rz    = pd.read_csv(OUTPUTS / "redzone_efficiency.csv")
drive = pd.read_csv(OUTPUTS / "drive_efficiency.csv").query("posteam != 'NA'")

SEASONS      = [2021, 2022, 2023]
COLORS       = {2021: "#5B9BD5", 2022: "#ED7D31", 2023: "#70AD47"}
TEMPLATE     = "plotly_dark"
FONT_FAMILY  = "Arial, sans-serif"

def base_layout(title, height=750):
    return dict(
        title=dict(text=title, font=dict(size=16, family=FONT_FAMILY), x=0.5, xanchor="center"),
        template=TEMPLATE,
        height=height,
        font=dict(family=FONT_FAMILY, size=11),
        margin=dict(l=80, r=40, t=60, b=40),
        paper_bgcolor="#1a1a2e",
        plot_bgcolor="#16213e",
    )

print("All metrics loaded.")

## EPA per Play by Team

In [3]:
fig = make_subplots(rows=1, cols=3, subplot_titles=["2021", "2022", "2023"], shared_xaxes=False)

for i, season in enumerate(SEASONS):
    df = epa[epa.season == season].sort_values("epa_per_play", ascending=True)
    bar_colors = [COLORS[season] if v >= 0 else "#c0392b" for v in df.epa_per_play]
    fig.add_trace(
        go.Bar(
            x=df.epa_per_play,
            y=df.posteam,
            orientation="h",
            marker=dict(color=bar_colors, line=dict(width=0)),
            showlegend=False,
            hovertemplate="<b>%{y}</b><br>EPA/Play: %{x:.4f}<extra></extra>",
        ),
        row=1, col=i + 1,
    )
    fig.add_vline(x=0, line=dict(color="rgba(255,255,255,0.3)", width=1), row=1, col=i + 1)

fig.update_layout(**base_layout("Expected Points Added per Play by Team  |  2021-2023", height=820))
fig.update_xaxes(title_text="EPA / Play", tickformat=".3f", showgrid=True, gridcolor="rgba(255,255,255,0.07)")
fig.update_yaxes(tickfont=dict(size=9))
fig.show()

## Success Rate by Team

In [ ]:
fig = make_subplots(rows=1, cols=3, subplot_titles=["2021", "2022", "2023"], shared_xaxes=True)

for i, season in enumerate(SEASONS):
    df = sr[sr.season == season].sort_values("success_rate", ascending=True)
    fig.add_trace(
        go.Bar(
            x=df.success_rate,
            y=df.posteam,
            orientation="h",
            marker=dict(color=COLORS[season], line=dict(width=0)),
            showlegend=False,
            hovertemplate="<b>%{y}</b><br>Success Rate: %{x:.1%}<extra></extra>",
        ),
        row=1, col=i + 1,
    )
    #50% baseline reference line across all subplots
    fig.add_vline(x=0.5, line=dict(color="rgba(255,255,255,0.25)", width=1, dash="dot"), row=1, col=i + 1)

fig.update_layout(**base_layout("Offensive Success Rate by Team  |  2021-2023  (Play is successful if EPA > 0)", height=820))
fig.update_xaxes(title_text="Success Rate", tickformat=".0%", showgrid=True, gridcolor="rgba(255,255,255,0.07)", range=[0.3, 0.56])
fig.update_yaxes(tickfont=dict(size=9))
fig.show()

## Win Probability Added (WPA) by Team

In [ ]:
fig = make_subplots(rows=1, cols=3, subplot_titles=["2021", "2022", "2023"], shared_xaxes=False)

for i, season in enumerate(SEASONS):
    df = wpa[wpa.season == season].sort_values("total_wpa", ascending=True)
    bar_colors = [COLORS[season] if v >= 0 else "#c0392b" for v in df.total_wpa]
    fig.add_trace(
        go.Bar(
            x=df.total_wpa,
            y=df.posteam,
            orientation="h",
            marker=dict(color=bar_colors, line=dict(width=0)),
            showlegend=False,
            hovertemplate="<b>%{y}</b><br>Total WPA: %{x:.3f}<extra></extra>",
        ),
        row=1, col=i + 1,
    )
    fig.add_vline(x=0, line=dict(color="rgba(255,255,255,0.3)", width=1), row=1, col=i + 1)

fig.update_layout(**base_layout("Total Win Probability Added by Team  |  2021-2023", height=820))
fig.update_xaxes(title_text="Total WPA", tickformat=".2f", showgrid=True, gridcolor="rgba(255,255,255,0.07)")
fig.update_yaxes(tickfont=dict(size=9))
fig.show()

## Red Zone TD Rate by Team

In [ ]:
fig = make_subplots(rows=1, cols=3, subplot_titles=["2021", "2022", "2023"], shared_xaxes=True)

for i, season in enumerate(SEASONS):
    df = rz[rz.season == season].sort_values("rz_td_rate", ascending=True)
    fig.add_trace(
        go.Bar(
            x=df.rz_td_rate,
            y=df.posteam,
            orientation="h",
            marker=dict(color=COLORS[season], line=dict(width=0)),
            showlegend=False,
            hovertemplate="<b>%{y}</b><br>RZ TD Rate: %{x:.1%}<br>TDs: %{customdata[0]} / %{customdata[1]} plays<extra></extra>",
            customdata=df[["rz_tds", "rz_plays"]].values,
        ),
        row=1, col=i + 1,
    )
    league_avg = rz[rz.season == season].rz_td_rate.mean()
    fig.add_vline(x=league_avg, line=dict(color="rgba(255,255,255,0.3)", width=1, dash="dot"), row=1, col=i + 1)

fig.update_layout(**base_layout("Red Zone TD Rate by Team  |  2021-2023  (dotted line = league avg)", height=820))
fig.update_xaxes(title_text="TD Rate (inside opponent 20)", tickformat=".0%", showgrid=True, gridcolor="rgba(255,255,255,0.07)", range=[0.08, 0.30])
fig.update_yaxes(tickfont=dict(size=9))
fig.show()

## Drive Efficiency

In [ ]:
fig = go.Figure()

for season in SEASONS:
    df = drive[drive.season == season].copy()
    fig.add_trace(go.Scatter(
        x=df.scoring_rate,
        y=df.avg_drive_epa,
        mode="markers+text",
        name=str(season),
        text=df.posteam,
        textposition="top center",
        textfont=dict(size=8),
        marker=dict(size=8, color=COLORS[season], opacity=0.85, line=dict(width=0.5, color="white")),
        hovertemplate="<b>%{text}</b>  %{fullData.name}<br>Scoring Rate: %{x:.1%}<br>Avg Drive EPA: %{y:.3f}<extra></extra>",
    ))

#quadrant reference lines computed from league-wide means across all seasons
mean_sr  = drive.scoring_rate.mean()
mean_epa = drive.avg_drive_epa.mean()
fig.add_hline(y=mean_epa, line=dict(color="rgba(255,255,255,0.2)", width=1, dash="dot"))
fig.add_vline(x=mean_sr,  line=dict(color="rgba(255,255,255,0.2)", width=1, dash="dot"))

layout = base_layout("Drive Efficiency: TD Scoring Rate vs Avg EPA per Drive  |  2021-2023", height=620)
layout.update(dict(
    xaxis=dict(title="TD Scoring Rate per Drive", tickformat=".0%", showgrid=True, gridcolor="rgba(255,255,255,0.07)"),
    yaxis=dict(title="Avg EPA per Drive", tickformat=".2f", showgrid=True, gridcolor="rgba(255,255,255,0.07)"),
    legend=dict(title="Season", bgcolor="rgba(0,0,0,0.3)"),
))
fig.update_layout(**layout)
fig.show()

## Combined: EPA vs Success Rate

In [ ]:
combined = epa.merge(sr, on=["season", "posteam"])

fig = go.Figure()

for season in SEASONS:
    df = combined[combined.season == season]
    fig.add_trace(go.Scatter(
        x=df.success_rate,
        y=df.epa_per_play,
        mode="markers+text",
        name=str(season),
        text=df.posteam,
        textposition="top center",
        textfont=dict(size=8),
        marker=dict(size=8, color=COLORS[season], opacity=0.85, line=dict(width=0.5, color="white")),
        hovertemplate="<b>%{text}</b>  %{fullData.name}<br>Success Rate: %{x:.1%}<br>EPA/Play: %{y:.4f}<extra></extra>",
    ))

mean_sr_all  = combined.success_rate.mean()
mean_epa_all = combined.epa_per_play.mean()
fig.add_hline(y=mean_epa_all, line=dict(color="rgba(255,255,255,0.2)", width=1, dash="dot"))
fig.add_vline(x=mean_sr_all,  line=dict(color="rgba(255,255,255,0.2)", width=1, dash="dot"))

layout = base_layout("EPA per Play vs Success Rate  |  2021-2023", height=620)
layout.update(dict(
    xaxis=dict(title="Success Rate", tickformat=".0%", showgrid=True, gridcolor="rgba(255,255,255,0.07)"),
    yaxis=dict(title="EPA per Play", tickformat=".3f", showgrid=True, gridcolor="rgba(255,255,255,0.07)"),
    legend=dict(title="Season", bgcolor="rgba(0,0,0,0.3)"),
))
fig.update_layout(**layout)
fig.show()